# Team 10 · Member 8 · Late fusion on the team split (CASMI 2026)

**Architecture 1, late fusion.** Everything trains inside this notebook, so it needs no other member's outputs.
Validation uses **the same split as Member 3 and Member 7**: Member 3's `make_split` (salt `casmi-split-v1`,
classes 1 and 2, np-examples proxy), `max_query_spectra=16`, `sample_molecules(..., 3000)`, a 10 ppm window and
Member 3's candidate pool and `evaluate`. The numbers are therefore directly comparable with B1 (0.264),
B2 (0.484) and Member 7's runs.

| Source | What it scores |
|---|---|
| `library` | cosine between each query spectrum and a candidate's centroid spectrum (same ion mode), mean over the molecule's spectra |
| `fp_ll` | log-likelihood of the candidate's Morgan bits under the MLP's predicted bit probabilities |
| `fp_tani` | soft Tanimoto between predicted probabilities and candidate bits |
| **fusion** | per-molecule z-score or reciprocal-rank normalisation, convex weights tuned on balanced MRR@25; `gated` = one weight set per library-confidence bin |

Fusion numbers are **2-fold cross-fitted**: each molecule is scored with weights tuned on the other half.

**Before you run: Add Input**
1. The competition data (Enveda CASMI 2026).
2. The code dataset `casmi-m8-code` (**new version**, it must contain `baselines/b2_library_search/casmi_pipeline.py`).
3. Recommended: the COCONUT and PubChemLite datasets Member 7 used (their path must contain `coconut` / `pubchem`).
   Without them the pool is training structures only, and the leaderboard score is capped at about 0.14.
4. RDKit is preinstalled on the current Kaggle image. If `import rdkit` fails, attach an RDKit wheel dataset.

**Settings:** Accelerator **GPU** (T4 or P100), **Internet off** (required for *Submit*).
Expected runtime is about 45-75 min with the extra candidates and about 30 min without them. Set `QUICK = True` for a 10-minute smoke test.

**Outputs** (`/kaggle/working`): `submission.csv`, plus `m8_results/` (summary.json, validation_report.csv, per-class tables,
fingerprint training history, figures, validation and test score tables, and the trained MLP weights).

In [ ]:
# ---- Settings ----
import glob
import os
import sys
import time

QUICK = False                       # True: small smoke run (300 val molecules, 2 epochs, no extra candidates)
N_VAL = 300 if QUICK else 3000      # validation molecules, as Member 7
MAX_QUERY_SPECTRA = 16              # Member 7: cp.make_split(meta, max_query_spectra=16)
PPM = 10.0                          # mass window, as Member 3 / Member 7
PER_STRUCTURE = 2 if QUICK else 8   # fingerprint-training spectra per structure (random, Member 3's select_train_rows)
MAX_TRAIN = 1_200_000               # cap on fingerprint-training spectra (memory)
FP_EPOCHS = 2 if QUICK else 20
FP_MAX_MIN = 150                    # time budget for MLP training
USE_EXTRA = not QUICK               # COCONUT + PubChemLite candidates when attached
OUT = "/kaggle/working"
RES = f"{OUT}/m8_results"
os.makedirs(RES, exist_ok=True)
T0 = time.time()


def stamp(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

In [ ]:
# ---- Code dataset: Member 3's casmi_pipeline.py (split, pool, evaluation) + Member 8's ensemble_common ----
cp_hits = glob.glob("/kaggle/input/**/baselines/b2_library_search/casmi_pipeline.py", recursive=True)
ens_hits = glob.glob("/kaggle/input/**/models/ensembles/ensemble_common/__init__.py", recursive=True)
if not cp_hits or not ens_hits:
    raise FileNotFoundError("Attach the new version of the casmi-m8-code dataset (needs "
                            "baselines/b2_library_search/casmi_pipeline.py and models/ensembles/ensemble_common).")
sys.path.insert(0, os.path.dirname(cp_hits[0]))
sys.path.insert(0, os.path.dirname(os.path.dirname(ens_hits[0])))
print("casmi_pipeline:", cp_hits[0])
print("ensemble_common:", os.path.dirname(ens_hits[0]))

In [ ]:
# ---- Imports, RDKit, GPU ----
import importlib
import json
import subprocess

try:
    import rdkit  # noqa: F401
except ImportError:
    wheels = glob.glob("/kaggle/input/**/rdkit*.whl", recursive=True)
    if wheels:  # offline: wheel attached as a dataset
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", wheels[0]], check=True)
    else:       # needs internet; submissions run offline, so attach a wheel if this branch is ever hit
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)
    importlib.invalidate_caches()

import casmi_pipeline as cp
import numpy as np
import pandas as pd
import rdkit
import torch
from ensemble_common.standalone import teamsplit as T
from ensemble_common.standalone.candidates import find_candidate_files, load_external, pool_fingerprints
from ensemble_common.standalone.fpnet import PackedFingerprintMLP

CFG = T.FusionConfig(ppm=PPM, n_random=100 if QUICK else 300)
DATA = cp.find_data_dir()
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("rdkit", rdkit.__version__, "| torch", torch.__version__, "| GPU:", GPU, "| CPUs:", os.cpu_count())
print("data:", DATA)
if GPU is None:
    print("WARNING: no GPU. Turn on the GPU accelerator, or the MLP will be very slow.")

In [ ]:
# ---- Member 3's split (exactly as Member 7 calls it), candidate pool, fingerprints ----
meta = cp.load_meta(DATA)
queries, lib_mask = cp.make_split(meta, max_query_spectra=MAX_QUERY_SPECTRA)
queries = cp.sample_molecules(queries, N_VAL)
truth = cp.truth_table(queries).reset_index(drop=True)
assert not lib_mask[queries.row_id.to_numpy()].any(), "validation spectra leaked into the library"
stamp(f"split: {len(truth):,} molecules (class 1: {(truth.cls == 1).sum()}, class 2: {(truth.cls == 2).sum()}, "
      f"np proxy: {(truth.query_lib == 'enveda-np-examples').sum()}), {len(queries):,} spectra")

import ensemble_common.standalone.candidates as candidates

candidates.SOURCES = ("coconut", "pubchem")  # any attached path naming COCONUT or PubChem(Lite)
files = find_candidate_files("/kaggle/input") if USE_EXTRA else {}
print("extra candidate files:", {k: [os.path.basename(f) for f in v] for k, v in files.items()} or "none")
if USE_EXTRA and not files:
    print("WARNING: COCONUT / PubChemLite not attached. The pool is training structures only (LB about 0.14).")
extra, extra_fp = load_external(files, n_bits=cp.FP_BITS, log=stamp) if files else (None, None)
pool = cp.build_pool(meta, lib_mask, extra_candidates=extra)
pool_fp = pool_fingerprints(pool, extra, extra_fp, cp.FP_BITS)
key_index = pd.Series(np.arange(len(pool)), index=pool.key.to_numpy())
n_train_struct = int((pool.formula.astype(str) != "").sum())
stamp(f"pool: {len(pool):,} structures ({n_train_struct:,} training, {len(pool) - n_train_struct:,} external)")

In [ ]:
# ---- One pass over train.parquet: centroid libraries + binned spectra for MLP training and validation ----
test, test_path = cp.load_test(DATA)
test_mols = test.molecule_id.drop_duplicates().tolist()
val_mass = queries.groupby("molecule_id").neutral_mass.median().reindex(truth.molecule_id)
test_mass = test.groupby("molecule_id").neutral_mass.median().reindex(test_mols)

train_rows = cp.select_train_rows(meta, lib_mask, per_structure=PER_STRUCTURE)
train_rows = train_rows[train_rows.inchikey14.isin(key_index.index)]
if len(train_rows) > MAX_TRAIN:
    train_rows = train_rows.sample(MAX_TRAIN, random_state=2026).sort_values("row_id")
train_rows = train_rows.reset_index(drop=True)

libs = T.build_libraries(cp, meta, lib_mask, pool, val_mass, test_mass, CFG)
print(f"centroids: validation {libs.val.sums.shape[0] // 2:,} structures, test {libs.test.sums.shape[0] // 2:,}")
mats = T.feature_pass(os.path.join(DATA, "train.parquet"), len(meta), libs,
                      {"train": train_rows.row_id.to_numpy(), "val": queries.row_id.to_numpy()}, CFG, log=stamp)
X_te = T.bin_file(test_path, len(test), CFG)
S_tr, S_va, S_te = (T.side_features(cp, f) for f in (train_rows, queries, test))
stamp(f"features: train {mats['train'].shape}, val {mats['val'].shape}, test {X_te.shape} "
      f"({test.molecule_id.nunique()} test molecules)")

In [ ]:
# ---- Fingerprint MLP (binned spectrum + precursor + ion mode + adduct -> 2048 Morgan bits) ----
val_t = queries.true_key.map(key_index)
ok = val_t.notna().to_numpy()  # validation BCE is logged only, never used to pick a checkpoint
model = PackedFingerprintMLP(epochs=FP_EPOCHS, max_minutes=FP_MAX_MIN)
model.fit(mats["train"], S_tr, train_rows.inchikey14.map(key_index).to_numpy(), pool_fp,
          val=(mats["val"][ok], S_va[ok], val_t[ok].astype(int).to_numpy()), log=stamp)
pd.DataFrame(model.history).to_csv(f"{RES}/fp_history.csv", index=False)
torch.save({k: v.cpu() for k, v in model.net.state_dict().items()}, f"{RES}/fp_mlp.pt")
P_va = model.predict(mats["val"], S_va)
P_te = model.predict(X_te, S_te)
del mats["train"]
import gc

gc.collect()
torch.cuda.empty_cache()
stamp("MLP trained")

In [ ]:
# ---- Validation: score the three sources, tune the fusion, cross-fit, compare with B1 ----
sc_val = T.score_molecules(cp, queries, mats["val"], P_va, libs.val, pool, pool_fp, CFG,
                           mols=truth.molecule_id.tolist(),
                           truth_keys=dict(zip(truth.molecule_id, truth.true_key, strict=True)))
stamp("validation scored")
rep = T.tune_and_evaluate(cp, sc_val, truth, pool, CFG, log=stamp)
b1 = {m: cp.rank_b1(pool, val_mass[m], PPM) for m in truth.molecule_id}
T.add_row(rep, "B1 (mass only)", cp.evaluate(b1, truth, pool.key.to_numpy(), ci=True), "reference")
val_cov = T.coverage(sc_val, truth)
pd.set_option("display.width", 200)
print("Validation MRR@25 on the team split (fusion rows are cross-fitted):")
print(rep["summary"].to_string(index=False))
print("coverage:", json.dumps(val_cov))
print("chosen for the submission:", rep["best"], json.dumps(T.describe(rep["params"][rep["best"]]), default=str))

In [ ]:
# ---- Test: same sources with the full-training library, chosen fusion, submission.csv ----
try:
    sc_te = T.score_molecules(cp, test, X_te, P_te, libs.test, pool, pool_fp, CFG, mols=test_mols)
    f_te = T.apply_best(rep, sc_te, CFG)
    test_rank = T.rankings(cp, sc_te, f_te, pool)
    test_cov = T.coverage(sc_te)
    T.score_table(sc_te, pool, f_te).to_parquet(f"{RES}/test_scores.parquet", index=False)
except Exception as err:  # noqa: BLE001 - the hidden-test rerun must always write a submission
    print("test scoring failed, using B1 for every molecule:", repr(err))
    test_rank, test_cov = {}, {"error": repr(err)}
fallback = [m for m in test_mols if not test_rank.get(m)]
for m in fallback:
    test_rank[m] = cp.rank_b1(pool, test_mass[m], PPM)
sub = cp.write_submission(test_rank, pool, test, path=f"{OUT}/submission.csv")
stamp(f"submission.csv: {len(sub)} rows, {len(fallback)} molecules on the B1 fallback | test coverage {test_cov}")

In [ ]:
# ---- Save results for the report and slides ----
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

best = rep["best"]
rep["summary"].to_csv(f"{RES}/validation_report.csv", index=False)
for name, table in rep["tables"].items():
    table.to_csv(f"{RES}/val_{name.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
val_fused = rep["val_scores"].get(best, sc_val.src.get(best))
if val_fused is not None:
    T.score_table(sc_val, pool, val_fused).to_parquet(f"{RES}/val_scores.parquet", index=False)
summary = {
    "date": time.strftime("%Y-%m-%d %H:%M"), "runtime_min": round((time.time() - T0) / 60, 1), "gpu": GPU,
    "quick": QUICK, "chosen": best, "chosen_params": T.describe(rep["params"][best]),
    "all_params": {k: T.describe(v) for k, v in rep["params"].items()},
    "split": {"function": "casmi_pipeline.make_split", "salt": "casmi-split-v1",
              "max_query_spectra": MAX_QUERY_SPECTRA, "molecules": int(len(truth)),
              "class1": int((truth.cls == 1).sum()), "class2": int((truth.cls == 2).sum()),
              "np_proxy": int((truth.query_lib == "enveda-np-examples").sum()), "spectra": int(len(queries))},
    "pool": {"size": int(len(pool)), "training_structures": n_train_struct, "external": int(len(pool) - n_train_struct),
             "extra_files": {k: [os.path.basename(f) for f in v] for k, v in files.items()}},
    "coverage": {"val": val_cov, "test": test_cov, "test_b1_fallback": len(fallback)},
    "validation": rep["summary"].set_index("method")[["balanced", "class 1", "class 2", "np proxy"]].to_dict("index"),
    "fp_history": model.history[-3:],
    "config": {"ppm": PPM, "n_val": N_VAL, "per_structure": PER_STRUCTURE, "max_train": MAX_TRAIN,
               "train_spectra": int(len(train_rows)), "fp_epochs": FP_EPOCHS, "fusion": vars(CFG)},
}
with open(f"{RES}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

s = rep["summary"].set_index("method")[["class 1", "class 2", "balanced"]].astype(float)
ax = s.iloc[::-1].plot.barh(figsize=(8, 0.45 * len(s) + 2), color=["#2a78d6", "#eb6834", "#555555"])
ax.set_xlabel("validation MRR@25 (team split; fusion rows cross-fitted)")
ax.set_title("Member 8 late fusion vs its sources and B1", loc="left")
plt.tight_layout()
plt.savefig(f"{RES}/validation_report.png", dpi=150)
plt.show()
h = pd.DataFrame(model.history)
ax = h.plot(x="epoch", y=[c for c in ("train_bce", "val_bce") if c in h], marker="o", figsize=(7, 4))
ax.set_ylabel("BCE")
plt.tight_layout()
plt.savefig(f"{RES}/fp_training.png", dpi=150)
plt.show()
for root, _, names in os.walk(OUT):
    for n in sorted(names):
        p = os.path.join(root, n)
        print(f"{os.path.getsize(p) / 1e6:8.2f} MB  {p}")
stamp("done")